## Contexte

**Getaround** est une plateforme de location de voitures entre particuliers (modèle Airbnb).  
5 millions d'utilisateurs · 20 000 voitures disponibles

**Problème :** les retards au checkout génèrent de la friction pour les conducteurs suivants.  
**Solution envisagée :** délai minimum entre deux locations consécutives.

**Types de checkin :** `connect` (smartphone) · `mobile` (app native)

---

**Livrables :**
1. Modèle ML de prédiction du prix de location (Partie 1)
2. Analyse des retards pour déterminer le seuil optimal (Partie 2)
3. API FastAPI `/predict` (POST JSON) + `/docs` (Swagger)
4. Dashboard Streamlit : slider threshold, CA et résas impactées


## Données

| Fichier | Onglet | Contenu |
|---|---|---|
| `data/get_around_pricing_project.csv` | — | 4 843 véhicules · 14 features · target : `rental_price_per_day` |
| `data/get_around_delay_analysis.xlsx` | `rentals_data` | 21 310 locations · checkin type, délai retard, location précédente |
| `data/get_around_delay_analysis.xlsx` | `Documentation` | Descriptif des champs de `rentals_data` |


# Getaround

In [1]:
import os
import re
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline 
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import mlflow
import mlflow.sklearn

load_dotenv()


True

# Partie 1 : EDA

Exploration des 2 fichiers fournis
- **get_around_princing_project.csv** : contient le pricing des véhicules en fonction de leurs caractéristiques
- **get_around_delay_analysis.xlsx** : contient l'analyse des retards lors du retour des véhicules


## 1.1) Analyse du fichier pour l'optimisation du pricing

In [2]:
df = pd.read_csv('data/get_around_pricing_project.csv', index_col=0)

In [3]:
df.head(10)

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
1,Citroën,13929,317,petrol,grey,convertible,True,True,False,False,False,True,True,264
2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183
5,Citroën,152352,225,petrol,black,convertible,True,True,False,False,True,True,True,131
6,Citroën,205219,145,diesel,grey,convertible,True,True,False,False,True,True,True,111
7,Citroën,115560,105,petrol,white,convertible,True,True,False,False,False,True,True,78
8,Peugeot,123886,125,petrol,black,convertible,True,False,False,False,False,True,True,79
9,Citroën,139541,135,diesel,white,convertible,False,False,False,False,True,False,True,132


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4843 entries, 0 to 4842
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   model_key                  4843 non-null   object
 1   mileage                    4843 non-null   int64 
 2   engine_power               4843 non-null   int64 
 3   fuel                       4843 non-null   object
 4   paint_color                4843 non-null   object
 5   car_type                   4843 non-null   object
 6   private_parking_available  4843 non-null   bool  
 7   has_gps                    4843 non-null   bool  
 8   has_air_conditioning       4843 non-null   bool  
 9   automatic_car              4843 non-null   bool  
 10  has_getaround_connect      4843 non-null   bool  
 11  has_speed_regulator        4843 non-null   bool  
 12  winter_tires               4843 non-null   bool  
 13  rental_price_per_day       4843 non-null   int64 
dtypes: bool(7), i

**Pas de valeurs null**

In [5]:
df.describe(include="all")

,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
count,4843,4.843000e+03,4843.00000,4843,4843,4843,4843,4843,4843,4843,4843,4843,4843,4843.000000
unique,28,NaN,NaN,4,10,8,2,2,2,2,2,2,2,NaN
top,Citroën,NaN,NaN,diesel,black,estate,True,True,False,False,False,False,True,NaN
freq,969,NaN,NaN,4641,1633,1606,2662,3839,3865,3881,2613,3674,4514,NaN
mean,NaN,1.409628e+05,128.98823,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,121.214536
std,NaN,6.019674e+04,38.99336,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.568268
min,NaN,-6.400000e+01,0.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.000000
25%,NaN,1.029135e+05,100.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,104.000000
50%,NaN,1.410800e+05,120.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,119.000000
75%,NaN,1.751955e+05,135.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136.000000


In [6]:
df.isna().any()

model_key                    False
mileage                      False
engine_power                 False
fuel                         False
paint_color                  False
car_type                     False
private_parking_available    False
has_gps                      False
has_air_conditioning         False
automatic_car                False
has_getaround_connect        False
has_speed_regulator          False
winter_tires                 False
rental_price_per_day         False
dtype: bool

In [7]:
#print(df.corr())

#fig = px.scatter_matrix(data_frame=df)
#fig.update_layout(height=1000, width=1000)
#fig.show()

In [8]:
# Recherche d'outliers potentiels
print("--- Valeurs problématiques ---")
print(f"mileage < 0        : {(df['mileage'] < 0).sum()} ligne(s)")
print(f"engine_power == 0  : {(df['engine_power'] == 0).sum()} ligne(s)")

print("\n--- mileage < 0 ---")
print(df[df['mileage'] < 0][['model_key', 'fuel', 'mileage', 'engine_power', 'rental_price_per_day']])

print("\n--- engine_power == 0 ---")
print(df[df['engine_power'] == 0][['model_key', 'fuel', 'mileage', 'engine_power', 'rental_price_per_day']])

print("\n--- Top 5 kilométrage ---")
print(df.nlargest(5, 'mileage')[['model_key', 'fuel', 'mileage', 'rental_price_per_day']])

print("\n--- Top 5 engine_power ---")
print(df.nlargest(5, 'engine_power')[['model_key', 'fuel', 'engine_power', 'rental_price_per_day']])

--- Valeurs problématiques ---
mileage < 0        : 1 ligne(s)
engine_power == 0  : 1 ligne(s)

--- mileage < 0 ---
     model_key    fuel  mileage  engine_power  rental_price_per_day
2938   Renault  diesel      -64           230                   274

--- engine_power == 0 ---
     model_key    fuel  mileage  engine_power  rental_price_per_day
3765    Nissan  diesel    81770             0                   108

--- Top 5 kilométrage ---
     model_key    fuel  mileage  rental_price_per_day
3732   Citroën  diesel  1000376                    37
557    Renault  diesel   484615                    91
2350   Peugeot  diesel   477571                    35
2829      Audi  diesel   439060                    10
3198   Citroën  diesel   405816                    22

--- Top 5 engine_power ---
     model_key    fuel  engine_power  rental_price_per_day
4146    Suzuki  petrol           423                   287
3601      Mini  petrol           412                   204
1      Citroën  petrol       

In [9]:
df['model_key'].unique()

array(['Citroën', 'Peugeot', 'PGO', 'Renault', 'Audi', 'BMW', 'Ford',
       'Mercedes', 'Opel', 'Porsche', 'Volkswagen', 'KIA Motors',
       'Alfa Romeo', 'Ferrari', 'Fiat', 'Lamborghini', 'Maserati',
       'Lexus', 'Honda', 'Mazda', 'Mini', 'Mitsubishi', 'Nissan', 'SEAT',
       'Subaru', 'Suzuki', 'Toyota', 'Yamaha'], dtype=object)

In [10]:
df['car_type'].unique()

array(['convertible', 'coupe', 'estate', 'hatchback', 'sedan',
       'subcompact', 'suv', 'van'], dtype=object)

In [11]:
df['fuel'].unique()

array(['diesel', 'petrol', 'hybrid_petrol', 'electro'], dtype=object)

In [12]:
df['paint_color'].unique()

array(['black', 'grey', 'white', 'red', 'silver', 'blue', 'orange',
       'beige', 'brown', 'green'], dtype=object)

## 1.2) Nettoyage & préparation


In [13]:
# Retirer le véhicule avec kilométrage négatif (-64 km)
df_model = df[df['mileage'] >= 0].copy()
print(f"Outlier mileage retiré : {len(df) - len(df_model)} ligne(s)")

# engine_power = 0 : physiquement impossible -> à retirer
n_before = len(df_model)
df_model = df_model[df_model['engine_power'] > 0].copy()
print(f"Outlier engine_power=0 retiré : {n_before - len(df_model)} ligne(s)")

print(f"\nDataset final : {len(df_model)} lignes (sur {len(df)} initiaux)")

Outlier mileage retiré : 1 ligne(s)
Outlier engine_power=0 retiré : 1 ligne(s)

Dataset final : 4841 lignes (sur 4843 initiaux)


## 1.3) Analuyses et Visualisations


In [14]:
# Distribution du prix (target)
fig = px.histogram(df_model, x='rental_price_per_day', nbins=50,
                   title='Distribution du prix de location / jour')
fig.show()

# Prix par type de véhicule
fig2 = px.box(df_model, x='car_type', y='rental_price_per_day',
              title='Prix par type de véhicule', points='outliers')
fig2.show()

# Prix par carburant
fig3 = px.box(df_model, x='fuel', y='rental_price_per_day',
              title='Prix par type de carburant', points='outliers')
fig3.show()

# Corrélations numériques
df_model[['mileage', 'engine_power', 'rental_price_per_day']].corr()


,mileage,engine_power,rental_price_per_day
mileage,1.000000,-0.049659,-0.448054
engine_power,-0.049659,1.000000,0.625430
rental_price_per_day,-0.448054,0.625430,1.000000


## 1.4) Entraînement : Modèle de Régression linéaire


In [15]:
CATEGORICAL = ['model_key', 'car_type', 'fuel', 'paint_color']
NUMERICAL   = ['mileage', 'engine_power']
BINARY      = ['private_parking_available', 'has_gps', 'has_air_conditioning',
               'automatic_car', 'has_getaround_connect', 'has_speed_regulator', 'winter_tires']
TARGET = 'rental_price_per_day'

X = df_model[CATEGORICAL + NUMERICAL + BINARY]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train : {X_train.shape[0]} lignes  |  Test : {X_test.shape[0]} lignes")


Train : 3872 lignes  |  Test : 969 lignes


In [16]:
# Paramètres OneHotEncoder :
#   handle_unknown='ignore' : à l'inférence, une valeur inconnue (ex: nouvelle marque) génère une ligne de zéros au lieu de lever une erreur
#   sparse_output=False : retourne un array numpy dense (requis par ColumnTransformer pour certains estimateurs qui ne gèrent pas les matrices sparse)
#   drop='first' : supprime la 1re modalité de chaque feature catégorielle pour éviter la colinéarité parfaite avec l'intercept (dummy variable trap)
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'), CATEGORICAL),
    ('num', StandardScaler(), NUMERICAL),
    ('bin', 'passthrough', BINARY),
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression()),
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("--- LR full dataset ---")
print(f"R²  : {r2:.4f}")
print(f"MAE : {mae:.2f} €/jour")
print(f"RMSE: {rmse:.2f} €/jour")


--- LR full dataset ---
R²  : 0.7060
MAE : 12.11 €/jour
RMSE: 17.94 €/jour


**Conclusion** : Le résultat n'est pas extraordinaire. On peut tenter d'améliorer en enlevant des outliers.

## 1.5) Modèle Régression Linéraire avec suppression des outliers (règle des 3σ)

On applique la règle des **3 sigmas** (moy. ± 3σ, couvre 99.7% d'une distribution normale) pour retirer les véhicules aberrants sur `mileage` et `engine_power` (ex : 1 000 376 km).

### 1.5.1) Filtrage outliers sur champs mileage + engine_power

In [17]:
SIGMA_FACTOR = 3  # règle 3σ

# Calcul de tous les masques sur df_model original, puis application simultanée
mask = pd.Series(True, index=df_model.index)
for col in NUMERICAL:
    mu, sig = df_model[col].mean(), df_model[col].std()
    mask &= df_model[col] <= mu + SIGMA_FACTOR * sig

df_clean_out2 = df_model[mask].copy()
_n_removed = len(df_model) - len(df_clean_out2)
print(f"Outliers retirés (> mean+{SIGMA_FACTOR}σ) : {_n_removed} ligne(s) — dataset nettoyé : {len(df_clean_out2)} lignes")

# Ré-entraînement sur le dataset filtré
_X = df_clean_out2[CATEGORICAL + NUMERICAL + BINARY]
_y = df_clean_out2[TARGET]
_X_train, _X_test, _y_train, _y_test = train_test_split(_X, _y, test_size=0.2, random_state=42)

pipeline_out2 = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'), CATEGORICAL),
        ('num', StandardScaler(), NUMERICAL),
        ('bin', 'passthrough', BINARY),
    ])),
    ('model', LinearRegression()),
])
pipeline_out2.fit(_X_train, _y_train)
_y_pred = pipeline_out2.predict(_X_test)
r2_out2   = r2_score(_y_test, _y_pred)
mae_out2  = mean_absolute_error(_y_test, _y_pred)
rmse_out2 = np.sqrt(mean_squared_error(_y_test, _y_pred))

# Affichage des résultats
print("--- LR règle 3σ --- ")
print(f"R²  : {r2_out2:.4f}")
print(f"MAE : {mae_out2:.2f} €/jour")
print(f"RMSE: {rmse_out2:.2f} €/jour")


Outliers retirés (> mean+3σ) : 92 ligne(s) — dataset nettoyé : 4749 lignes
--- LR règle 3σ --- 
R²  : 0.6246
MAE : 12.88 €/jour
RMSE: 19.24 €/jour


/home/gviel/app/miniconda3/envs/getaround/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:241: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


**Conclusion 1** : après filtrage, on casse des colinéarités qui existaient dans le dataset complet (un ou plusieurs colonnes à zéro) ce qui produit alors des coefficients trop élevés -> il faut tester avec **Ridge** pour faire une régularisation L2 des coefficients.

In [18]:
def plot_pipeline_coefs(pipeline, categorical, numerical, binary, top_n=20):
    '''
        Affiche les top_n coefficients d'un pipeline LR/Ridge sous forme de barplot horizontal Plotly.
        Fonctionne avec tout pipeline sklearn contenant 'preprocessor' (ColumnTransformer) et 'model'.
    '''
    model = pipeline.named_steps['model']
    feature_names = (
        pipeline.named_steps['preprocessor']
        .named_transformers_['cat']
        .get_feature_names_out(categorical).tolist()
        + numerical + binary
    )
    coefs = (
        pd.Series(model.coef_, index=feature_names)
        .sort_values(key=abs, ascending=False)
        .head(top_n)
        .reset_index()
    )
    coefs.columns = ['feature', 'coef']

    fig = px.bar(
        coefs.sort_values('coef'),
        x='coef', y='feature',
        orientation='h',
        color='coef',
        color_continuous_scale='RdBu',
        color_continuous_midpoint=0,
        title=f"Top {top_n} coefficients — {type(model).__name__} (intercept={model.intercept_:.2f})",
        labels={'coef': 'Coefficient', 'feature': ''},
        height=max(400, top_n * 28),
    )
    fig.add_vline(x=0, line_color='black', line_width=1)
    fig.update_layout(coloraxis_showscale=False)
    fig.show()

    return coefs

In [19]:
coefs_lr_baseline = plot_pipeline_coefs(pipeline, CATEGORICAL, NUMERICAL, BINARY, 55)

In [20]:

coefs_lr_out2 = plot_pipeline_coefs(pipeline_out2, CATEGORICAL, NUMERICAL, BINARY, 55)

**Observation** : les deux graphiques montrent le même phénomène — **dummy variable trap**.

Sans `drop='first'` dans `OneHotEncoder`, toutes les modalités d'une même feature somment à 1 pour chaque ligne, ce qui est colinéaire avec l'intercept. `LinearRegression` produit alors des coefficients aberrants (~10¹⁴) pour les 50 colonnes OHE (`model_key`, `car_type`, `fuel`, `paint_color`).

Seuls les 5 features non-OHE ont des coefficients interprétables (mais écrasés visuellement par l'échelle) :
`engine_power` (+14 €/j/σ), `mileage` (−13), `has_gps` (+11), `has_getaround_connect` (+5), `has_speed_regulator` (+5).

→ Ridge en §1.5.2 régularise ces coefficients aberrants.


### 1.5.2) Filtrage outliers + régularisation par Ridge / Lasso

In [21]:
# Calcul de tous les masques sur df_model original, puis application simultanée
mask = pd.Series(True, index=df_model.index)
for col in NUMERICAL:
    mu, sig = df_model[col].mean(), df_model[col].std()
    mask &= df_model[col] <= mu + SIGMA_FACTOR * sig

df_clean_reg = df_model[mask].copy()
_n_removed = len(df_model) - len(df_clean_reg)
print(f"Outliers retirés (> mean+{SIGMA_FACTOR}σ) : {_n_removed} ligne(s) — dataset nettoyé : {len(df_clean_reg)} lignes")

# Ré-entraînement sur le dataset filtré
_X = df_clean_reg[CATEGORICAL + NUMERICAL + BINARY]
_y = df_clean_reg[TARGET]
_X_train, _X_test, _y_train, _y_test = train_test_split(_X, _y, test_size=0.2, random_state=42)

# Test avec Ridge
pipeline_reg = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL),
        ('num', StandardScaler(), NUMERICAL),
        ('bin', 'passthrough', BINARY),
    ])),
    ('model', Ridge(alpha=10.0)),
])
pipeline_reg.fit(_X_train, _y_train)
_y_pred = pipeline_reg.predict(_X_test)
r2_reg   = r2_score(_y_test, _y_pred)
mae_reg  = mean_absolute_error(_y_test, _y_pred)
rmse_reg = np.sqrt(mean_squared_error(_y_test, _y_pred))

# Affichage des résultats
print("--- Ridge + règle 3σ ---")
print(f"R² : {r2_reg:.4f}")
print(f"MAE : {mae_reg:.2f} €/jour")
print(f"RMSE: {rmse_reg:.2f} €/jour")

# Test avec Lasso
pipeline_lasso = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL),
        ('num', StandardScaler(), NUMERICAL),
        ('bin', 'passthrough', BINARY),
    ])),
    ('model', Lasso(alpha=0.1)),
])
pipeline_lasso.fit(_X_train, _y_train)
_y_pred = pipeline_lasso.predict(_X_test)
r2_lasso   = r2_score(_y_test, _y_pred)
mae_lasso  = mean_absolute_error(_y_test, _y_pred)
rmse_lasso = np.sqrt(mean_squared_error(_y_test, _y_pred))

# Affichage des résultats
print("--- Lasso + règle 3σ ---")
print(f"R²  : {r2_lasso:.4f}")
print(f"MAE : {mae_lasso:.2f} €/jour")
print(f"RMSE: {rmse_lasso:.2f} €/jour")

Outliers retirés (> mean+3σ) : 92 ligne(s) — dataset nettoyé : 4749 lignes
--- Ridge + règle 3σ ---
R² : 0.6317
MAE : 12.77 €/jour
RMSE: 19.06 €/jour
--- Lasso + règle 3σ ---
R²  : 0.6277
MAE : 12.80 €/jour
RMSE: 19.17 €/jour


## 1.6) Régression Linéraire en supprimant feature paint_color (dataset complet)

In [22]:
# Ré-entraînement sur le dataset sans paint_color
CATEGORICAL_WITHOUT_PAINT = [ f for f in CATEGORICAL if f != 'paint_color']

df_clean_paint = df_model.copy()

_X = df_clean_paint[CATEGORICAL_WITHOUT_PAINT + NUMERICAL + BINARY]
_y = df_clean_paint[TARGET]
_X_train, _X_test, _y_train, _y_test = train_test_split(_X, _y, test_size=0.2, random_state=42)

pipeline_paint = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'), CATEGORICAL_WITHOUT_PAINT),
        ('num', StandardScaler(), NUMERICAL),
        ('bin', 'passthrough', BINARY),
    ])),
    ('model', LinearRegression()),
])
pipeline_paint.fit(_X_train, _y_train)
_y_pred = pipeline_paint.predict(_X_test)

r2_paint   = r2_score(_y_test, _y_pred)
mae_paint  = mean_absolute_error(_y_test, _y_pred)
rmse_paint = np.sqrt(mean_squared_error(_y_test, _y_pred))

# Affichage des résultats
print("--- LR without paint_color (full dataset) ---")
print(f"R²  : {r2_paint:.4f}")
print(f"MAE : {mae_paint:.2f} €/jour")
print(f"RMSE: {rmse_paint:.2f} €/jour")


--- LR without paint_color (full dataset) ---
R²  : 0.6977
MAE : 12.21 €/jour
RMSE: 18.19 €/jour


## 1.7) Utilisation d'un modèle XGBoost + GridSearchCV

In [23]:
# Pipeline XGBoost sur le dataset complet (même split que baseline)
_X = df_model[CATEGORICAL + NUMERICAL + BINARY]
_y = df_model[TARGET]

_X_train, _X_test, _y_train, _y_test = train_test_split(_X, _y, test_size=0.2, random_state=42)

pipeline_xgb = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL),
        ('num', StandardScaler(), NUMERICAL),
        ('bin', 'passthrough', BINARY),
    ])),
    ('model', XGBRegressor(random_state=42, n_jobs=6, verbosity=0)),
])

param_grid = {
    'model__n_estimators':  [100, 200, 300],
    'model__max_depth':     [3, 5, 7, 12],
    'model__learning_rate': [0.002, 0.05, 0.1, 0.2],
}

grid_search = GridSearchCV(
    pipeline_xgb,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)

print(f"\nMeilleurs hyperparamètres : {grid_search.best_params_}")
print(f"R² CV (validation)        : {grid_search.best_score_:.4f}")

pipeline_xgb = grid_search.best_estimator_
y_pred_xgb = pipeline_xgb.predict(X_test)
r2_xgb   = r2_score(y_test, y_pred_xgb)
mae_xgb  = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

# Affichage des résultats
print("--- XGBoost (GridSearchCV) without paint_color ---")
print(f"R²  : {r2_xgb:.4f}")
print(f"MAE : {mae_xgb:.2f} €/jour")
print(f"RMSE: {rmse_xgb:.2f} €/jour")

Fitting 5 folds for each of 48 candidates, totalling 240 fits

Meilleurs hyperparamètres : {'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__n_estimators': 200}
R² CV (validation)        : 0.7682
--- XGBoost (GridSearchCV) without paint_color ---
R²  : 0.7646
MAE : 10.39 €/jour
RMSE: 16.05 €/jour


## 1.8) Comparatif des modèles

Les 6 variantes testées sont comparées ci-dessous sur les métriques R², MAE et RMSE (test set).  
Le meilleur modèle est retenu automatiquement comme `best_pipeline`.


In [24]:
results = pd.DataFrame([
    {"Modèle": "LR baseline",          "Dataset": len(df_model),      "R²": r2,       "MAE": mae,       "RMSE": rmse},
    {"Modèle": "LR + 3σ",              "Dataset": len(df_clean_out2), "R²": r2_out2,  "MAE": mae_out2,  "RMSE": rmse_out2},
    {"Modèle": "Ridge + 3σ",           "Dataset": len(df_clean_reg),  "R²": r2_reg,   "MAE": mae_reg,   "RMSE": rmse_reg},
    {"Modèle": "Lasso + 3σ",           "Dataset": len(df_clean_reg),  "R²": r2_lasso, "MAE": mae_lasso, "RMSE": rmse_lasso},
    {"Modèle": "LR sans paint_color",  "Dataset": len(df_clean_paint),"R²": r2_paint, "MAE": mae_paint, "RMSE": rmse_paint},
    {"Modèle": "XGBoost GridSearchCV", "Dataset": len(df_model),      "R²": r2_xgb,   "MAE": mae_xgb,   "RMSE": rmse_xgb},
]).sort_values("R²", ascending=False).reset_index(drop=True)

results_display = results.copy()
results_display["R²"]   = results_display["R²"].map("{:.4f}".format)
results_display["MAE"]  = results_display["MAE"].map("{:.2f} €/j".format)
results_display["RMSE"] = results_display["RMSE"].map("{:.2f} €/j".format)
display(results_display)


,Modèle,Dataset,R²,MAE,RMSE
0,XGBoost GridSearchCV,4841,0.7646,10.39 €/j,16.05 €/j
1,LR baseline,4841,0.7060,12.11 €/j,17.94 €/j
2,LR sans paint_color,4841,0.6977,12.21 €/j,18.19 €/j
3,Ridge + 3σ,4749,0.6317,12.77 €/j,19.06 €/j
4,Lasso + 3σ,4749,0.6277,12.80 €/j,19.17 €/j
5,LR + 3σ,4749,0.6246,12.88 €/j,19.24 €/j


In [25]:
pipelines = {
    "LR baseline":          (pipeline,       r2),
    "LR + 3σ":              (pipeline_out2,  r2_out2),
    "Ridge + 3σ":           (pipeline_reg,   r2_reg),
    "Lasso + 3σ":           (pipeline_lasso, r2_lasso),
    "LR sans paint_color":  (pipeline_paint, r2_paint),
    "XGBoost GridSearchCV": (pipeline_xgb,   r2_xgb),
}
best_name, (best_pipeline, _) = max(pipelines.items(), key=lambda x: x[1][1])
print(f"→ Meilleur modèle retenu : {best_name}  (R²={pipelines[best_name][1]:.4f})")


→ Meilleur modèle retenu : XGBoost GridSearchCV  (R²=0.7646)


## 1.9) Sauvegarde du meilleur modèle (en local pour tests)

In [26]:
import joblib, os
os.makedirs('models', exist_ok=True)
joblib.dump(best_pipeline, 'models/getaround_pricing_model.pkl')
print("Modèle sauvegardé → models/getaround_pricing_model.pkl")


Modèle sauvegardé → models/getaround_pricing_model.pkl


## 1.10) Envoi du modèle vers MLFlow

> **Prérequis :** remplir le fichier `.env` à la racine du projet (copier `.env_template`, renseigner les clés AWS et le token HuggingFace) avant d'exécuter cette cellule.

In [ ]:
from mlflow.client import MlflowClient

MLFLOW_URI       = os.getenv('MLFLOW_URI',       'https://gviel-mlflow37.hf.space/')
EXPERIMENT_NAME  = os.getenv('EXPERIMENT_NAME',  'Get_Around')
MODEL_NAME       = os.getenv('MODEL_NAME',       'getaround_pricing')
MODEL_ENV        = os.getenv('MODEL_ENV',        'prod')    # prod | staging | test

# Résoudre les métriques et le type du meilleur modèle
pipelines_meta = {
    "LR baseline":          (pipeline,       r2,       mae,       rmse,       "LinearRegression", "lr"),
    "LR + 3σ":              (pipeline_out2,  r2_out2,  mae_out2,  rmse_out2,  "LR_3sigma", "lr_3s"),
    "Ridge + 3σ":           (pipeline_reg,   r2_reg,   mae_reg,   rmse_reg,   "Ridge_3sigma", "ridge_3s"),
    "Lasso + 3σ":           (pipeline_lasso, r2_lasso, mae_lasso, rmse_lasso, "Lasso_3sigma", "lasso_3s"),
    "LR sans paint_color":  (pipeline_paint, r2_paint, mae_paint, rmse_paint, "LR_no_paint", "lr_paint"),
    "XGBoost GridSearchCV": (pipeline_xgb,   r2_xgb,   mae_xgb,   rmse_xgb,   "XGBoost", "xgb"),
}
_, best_r2_val, best_mae_val, best_rmse_val, model_type, model_type_short = pipelines_meta[best_name]

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

with mlflow.start_run(experiment_id=experiment.experiment_id) as run:
    mlflow.log_params({
        "model":                model_type,
        "test_size":            0.2,
        "categorical_features": str(CATEGORICAL),
        "numerical_features":   str(NUMERICAL),
        "binary_features":      str(BINARY),
    })
    metrics = {"r2": best_r2_val, "mae": best_mae_val, "rmse": best_rmse_val}
    # Coefficients absolus (modèles linéaires uniquement)
    _step = best_pipeline.named_steps['model']
    if hasattr(_step, 'coef_'):
        _abs_coefs = np.abs(_step.coef_)
        metrics["coef_abs_min"] = float(_abs_coefs.min())
        metrics["coef_abs_max"] = float(_abs_coefs.max())
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(
        best_pipeline,
        name="model",
        registered_model_name=MODEL_NAME,
    )
    print(f"run_id  : {run.info.run_id}")
    print(f"model   : {MODEL_NAME} ({model_type})  |  experiment : {EXPERIMENT_NAME}")
    print(f"R²={best_r2_val:.4f}  MAE={best_mae_val:.2f} €/j  RMSE={best_rmse_val:.2f} €/j")
    if "coef_abs_min" in metrics:
        print(f"coef |min|={metrics['coef_abs_min']:.4f}  |max|={metrics['coef_abs_max']:.4f}")

# Tagging de la version enregistrée
# status calculé par comparaison du R² avec les versions existantes du même modèle
client = MlflowClient()
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest = sorted(versions, key=lambda v: int(v.version), reverse=True)[0]

other_versions = [v for v in versions if v.version != latest.version]
existing_r2s = []
for v in other_versions:
    try:
        r2_val = client.get_run(v.run_id).data.metrics.get("r2")
        if r2_val is not None:
            existing_r2s.append(r2_val)
    except Exception:
        pass

if not existing_r2s or best_r2_val > max(existing_r2s):
    status = "best"
elif best_r2_val < min(existing_r2s):
    status = "worst"
else:
    status = "challenger"

client.set_model_version_tag(MODEL_NAME, latest.version, "env",    MODEL_ENV)
client.set_model_version_tag(MODEL_NAME, latest.version, "status", status)
print(f"Tags appliqués → v{latest.version} : env={MODEL_ENV}, status={status}  ({len(existing_r2s)} version(s) existante(s) comparées)")

Successfully registered model 'getaround_pricing_xgb'.
2026/06/30 02:13:34 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: getaround_pricing_xgb, version 1
Created version '1' of model 'getaround_pricing_xgb'.


run_id  : 495558c41f844e58bee4a9ad5f3241e7
model   : getaround_pricing_xgb (XGBoost)  |  experiment : Get_Around
R²=0.7646  MAE=10.39 €/j  RMSE=16.05 €/j
🏃 View run serious-hare-330 at: https://gviel-mlflow37.hf.space/#/experiments/8/runs/495558c41f844e58bee4a9ad5f3241e7
🧪 View experiment at: https://gviel-mlflow37.hf.space/#/experiments/8
Tags appliqués → v1 : env=prod, status=best  (0 version(s) existante(s) comparées)


# Partie 2 : Analyse des retards

In [28]:
df_delay = pd.read_excel("data/get_around_delay_analysis.xlsx", sheet_name="rentals_data")

In [29]:
df_delay

,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
0,505000,363965,mobile,canceled,NaN,NaN,NaN
1,507750,269550,mobile,ended,-81.0,NaN,NaN
2,508131,359049,connect,ended,70.0,NaN,NaN
3,508865,299063,connect,canceled,NaN,NaN,NaN
4,511440,313932,mobile,ended,NaN,NaN,NaN
...,...,...,...,...,...,...,...
21305,573446,380069,mobile,ended,NaN,573429.0,300.0
21306,573790,341965,mobile,ended,-337.0,NaN,NaN
21307,573791,364890,mobile,ended,144.0,NaN,NaN
21308,574852,362531,connect,ended,-76.0,NaN,NaN


## 2.1) Exploration du dataset


In [30]:
print(f"Shape : {df_delay.shape}")
print(f"\nColonnes :\n{df_delay.dtypes}\n")
print(f"Valeurs null :\n{df_delay.isna().sum()}\n")
print(f"checkin_type :\n{df_delay['checkin_type'].value_counts()}\n")
print(f"state :\n{df_delay['state'].value_counts()}")


Shape : (21310, 7)

Colonnes :
rental_id                                       int64
car_id                                          int64
checkin_type                                   object
state                                          object
delay_at_checkout_in_minutes                  float64
previous_ended_rental_id                      float64
time_delta_with_previous_rental_in_minutes    float64
dtype: object

Valeurs null :
rental_id                                         0
car_id                                            0
checkin_type                                      0
state                                             0
delay_at_checkout_in_minutes                   4964
previous_ended_rental_id                      19469
time_delta_with_previous_rental_in_minutes    19469
dtype: int64

checkin_type :
checkin_type
mobile     17003
connect     4307
Name: count, dtype: int64

state :
state
ended       18045
canceled     3265
Name: count, dtype: int64


## 2.2) Identification des retards ayant impacté la location suivante

Un retard impacte la location suivante si `delay_at_checkout >= time_delta_with_previous_rental` de cette location suivante.

In [71]:
# On travaille uniquement sur les locations non annulées
df_ended = df_delay[df_delay['state'] != 'canceled'].copy()

# A  = location rendue EN RETARD        (delay_at_checkout_in_minutes > 0)
# B  = location SUIVANTE sur le même véhicule
#        → B.previous_ended_rental_id = A.rental_id
#        → B apporte : le délai prévu entre A et B  (B_time_delta)
#                      et le type de checkin de B   (B_checkin_type)
#
# Jointure : df_A  INNER JOIN  df_B  ON  A.rental_id = B.previous_ended_rental_id
# Un retard de A impacte B si : A.delay_at_checkout >= B.time_delta
# ───────────────────────────────────────────────────────────────────────────

# Table B : locations ayant une location précédente connue (not null)
df_B = df_ended[df_ended['previous_ended_rental_id'].notna()][[
    'previous_ended_rental_id',
    'time_delta_with_previous_rental_in_minutes',
    'checkin_type',
]].rename(columns={
    'previous_ended_rental_id':                   'rental_id',       # clé de jointure → A.rental_id
    'time_delta_with_previous_rental_in_minutes': 'B_time_delta',    # délai prévu entre A et B
    'checkin_type':                               'B_checkin_type',  # type de checkin de B
})

# Table A : locations terminées en retard
df_A = df_ended[df_ended['delay_at_checkout_in_minutes'] > 0]

# Jointure : on ne garde que les cas où A a une location suivante B connue
df_impact = df_A.merge(df_B, on='rental_id', how='inner')

# Le retard de A a impacté B si delay_A >= délai prévu avant B
df_impact['caused_problem'] = (
    df_impact['delay_at_checkout_in_minutes'] >= df_impact['B_time_delta']
)
df_caused = df_impact[df_impact['caused_problem']].copy()

n_total  = len(df_impact)
n_caused = len(df_caused)
print(f"Locations A en retard avec une location suivante B connue : {n_total}")
print(f"Dont ayant causé un problème à B (delay_A >= B_time_delta) : {n_caused}  ({n_caused/n_total*100:.1f}%)")

Locations A en retard avec une location suivante B connue : 767
Dont ayant causé un problème à B (delay_A >= B_time_delta) : 182  (23.7%)


## 2.3) Proportion connect / mobile parmi les retards ayant impacté la location suivante

In [72]:
# Retards de A ayant réellement impacté B (delay_A >= B_time_delta)
df_caused = df_impact[df_impact['caused_problem']].copy()

print(f"Locations concernées : {len(df_caused)}\n")

# Répartition connect / mobile (type de checkin de A)
counts = df_caused['checkin_type'].value_counts()
pcts   = df_caused['checkin_type'].value_counts(normalize=True).mul(100).round(1)
summary = pd.DataFrame({'nb': counts, 'pct_%': pcts})
print(summary)

# Bar chart
fig = px.bar(
    summary.reset_index(),
    x='checkin_type', y='nb',
    color='checkin_type',
    title='Retards de A ayant impacté B — connect vs mobile',
    labels={'nb': 'Nombre de cas', 'checkin_type': 'Type'},
    text='pct_%',
)
fig.update_traces(texttemplate='%{text:.1f} %', textposition='outside')
fig.show()

Locations concernées : 182

               nb  pct_%
checkin_type            
mobile        132   72.5
connect        50   27.5


## 2.4) Statistiques descriptives des retards par type


In [73]:
stats = df_caused.groupby('checkin_type')['delay_at_checkout_in_minutes'].agg(
    count='count',
    mean='mean',
    median='median',
    std='std',
    p75=lambda x: x.quantile(0.75),
    p90=lambda x: x.quantile(0.90),
    p95=lambda x: x.quantile(0.95),
    max='max',
).round(1)

print("Statistiques du délai de retard (en minutes) — retards ayant impacté la location suivante :")
print(stats.T)

Statistiques du délai de retard (en minutes) — retards ayant impacté la location suivante :
checkin_type  connect   mobile
count            50.0    132.0
mean             75.3    295.2
median           42.0     63.0
std              96.1   1219.8
p75              98.5    157.0
p90             173.1    399.0
p95             228.1    834.4
max             550.0  12968.0


**Conclusion**
- Les retards `connect` sont plus courts et plus concentrés (écart-type faible)
- tandis que les retards `mobile` présentent une moyenne et une variance nettement plus élevées (à cause de quelques cas extrêmes)
- Les médianes restent proches pour les deux types
- la majorité des conflits sont inférieurs à 1h, mais la "queue" de distribution `mobile` est bien plus longue

**Cela justifie de définir un délai minimum différencié selon le type de checkin.**

## 2.5) Visualisations des distributions des retards


In [74]:
# Plafonner à 600 min (10h) pour lisibilité — les valeurs extrêmes écraseront sinon l'axe
CAP = 600
df_viz = df_caused[df_caused['delay_at_checkout_in_minutes'] <= CAP].copy()

# Boxplot
fig1 = px.box(
    df_viz, x='checkin_type', y='delay_at_checkout_in_minutes',
    color='checkin_type', points='outliers',
    title=f'Distribution des retards impactants par type (≤ {CAP} min)',
    labels={'delay_at_checkout_in_minutes': 'Retard (min)', 'checkin_type': 'Type'},
)
fig1.show()

# Histogramme / distribution
fig2 = px.histogram(
    df_viz, x='delay_at_checkout_in_minutes',
    color='checkin_type', barmode='overlay',
    nbins=60, opacity=0.7,
    title=f'Distribution des retards impactants (≤ {CAP} min)',
    labels={'delay_at_checkout_in_minutes': 'Retard (min)', 'checkin_type': 'Type'},
)
fig2.show()

## 2.6) Analyse percentiles : estimation du délai optimal

Les percentiles permettent de choisir un seuil de délai minimum entre deux locations consécutives : un délai couvrant p% des retards évite p% des conflits.


In [75]:
percentiles = [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

rows = []
for checkin_type, grp in df_caused.groupby('checkin_type'):
    for p in percentiles:
        val = grp['delay_at_checkout_in_minutes'].quantile(p)
        rows.append({'checkin_type': checkin_type, 'percentile': f'p{int(p*100)}', 'délai_min': round(val)})

df_rows = pd.DataFrame(rows)

# Construction manuelle du tableau croisé (évite le bug pandas unstack/pivot)
types = df_rows['checkin_type'].unique()
df_pct = pd.DataFrame(
    {t: {r['percentile']: r['délai_min'] for r in rows if r['checkin_type'] == t} for t in types}
)
df_pct.index.name = 'percentile'
print("Délai (en minutes) couvrant X% des retards impactants :")
print(df_pct)

# Courbe percentile → délai
fig = px.line(
    df_rows, x='percentile', y='délai_min',
    color='checkin_type', markers=True,
    title='Délai nécessaire pour couvrir X% des retards impactants (connect vs mobile)',
    labels={'délai_min': 'Délai minimum (min)', 'percentile': 'Percentile', 'checkin_type': 'Type'},
)
fig.show()

Délai (en minutes) couvrant X% des retards impactants :
            connect  mobile
percentile                 
p50              42      63
p60              65      81
p70              91     122
p75              98     157
p80             113     208
p85             149     275
p90             173     399
p95             228     834
